<a href="https://colab.research.google.com/github/arunkumar-ns/AI-LAB/blob/main/1WN24CS050_Week6_PropositionalLogic_TruthTableEnumeration_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

import itertools
import re


def evaluate(expression, assignment):
    expression = expression.replace("<=>", "==")

    # Replace implication (A => B) with (not A or B)
    expression = re.sub(
        r'\(\s*([^()]+?)\s*=>\s*([^()]+?)\s*\)',
        r'(not (\1) or (\2))',
        expression
    )

    # Handle simple implications without parentheses
    expression = re.sub(
        r'\b([A-Z])\s*=>\s*(~?\s*[A-Z])\b',
        r'(not (\1) or (\2))',
        expression
    )

    # Replace logical operators with Python operators
    expression = expression.replace("^", " and ")
    expression = re.sub(r'\bv\b', ' or ', expression)
    expression = expression.replace("~", " not ")

    # Replace proposition symbols with Boolean values
    for symbol, value in assignment.items():
        expression = re.sub(
            r'\b' + re.escape(symbol) + r'\b',
            str(value),
            expression
        )

    return bool(eval(expression))


def get_symbols(kb, query):
    expressions = kb + " " + query
    symbols = set(re.findall(r'\b[A-Z]\b', expressions))
    return sorted(symbols)


def tt_check_all(kb, query, symbols, assignment, results):
    if not symbols:
        kb_value = evaluate(kb, assignment)
        query_value = evaluate(query, assignment)

        results.append(
            (assignment.copy(), kb_value, query_value)
        )

        if kb_value:
            return query_value

        return True

    symbol = symbols[0]
    rest = symbols[1:]

    assignment[symbol] = True
    result1 = tt_check_all(
        kb, query, rest, assignment, results
    )

    assignment[symbol] = False
    result2 = tt_check_all(
        kb, query, rest, assignment, results
    )

    del assignment[symbol]

    return result1 and result2


def tt_entails(kb, query):
    symbols = get_symbols(kb, query)
    results = []

    print("\n" + "=" * 65)
    print("             TRUTH TABLE ENTAILMENT")
    print("=" * 65)

    print("\nKnowledge Base:", kb)
    print("Query         :", query)
    print("Symbols       :", ", ".join(symbols))

    entailment = tt_check_all(
        kb, query, symbols, {}, results
    )

    print("\n" + "-" * 65)
    print("TRUTH TABLE")
    print("-" * 65)

    for symbol in symbols:
        print(f"{symbol:^10}", end="")

    print(f"{'KB':^12}{'QUERY':^12}")
    print("-" * 65)

    for assignment, kb_value, query_value in results:
        for symbol in symbols:
            print(
                f"{str(assignment[symbol]):^10}",
                end=""
            )

        print(
            f"{str(kb_value):^12}"
            f"{str(query_value):^12}"
        )

    print("-" * 65)

    print("\nRESULT:")

    if entailment:
        print("KB ENTAILS QUERY")
        print("KB |= QUERY")
    else:
        print("KB DOES NOT ENTAIL QUERY")
        print("KB |/= QUERY")

    print("\n" + "=" * 65)
    print("Name : Arun Kumar N S")
    print("USN  : 1WN24CS0050")
    print("=" * 65)


# Main program
kb = input("Enter Knowledge Base: ")
query = input("Enter Query: ")

tt_entails(kb, query)


Enter Knowledge Base: (Q => P) ^ (P => ~Q) ^ (Q v R)
Enter Query: R

             TRUTH TABLE ENTAILMENT

Knowledge Base: (Q => P) ^ (P => ~Q) ^ (Q v R)
Query         : R
Symbols       : P, Q, R

-----------------------------------------------------------------
TRUTH TABLE
-----------------------------------------------------------------
    P         Q         R          KB        QUERY    
-----------------------------------------------------------------
   True      True      True      False        True    
   True      True     False      False       False    
   True     False      True       True        True    
   True     False     False      False       False    
  False      True      True      False        True    
  False      True     False      False       False    
  False     False      True       True        True    
  False     False     False      False       False    
-----------------------------------------------------------------

RESULT:
KB ENTAILS QUERY
KB |= Q